# PCA + BiLSTM (Baseline Experiment)

**Challenge:** AN2DL 2024 — Pirate Pain Classification  
**Approach:** PCA dimensionality reduction on joint angles, then Bidirectional LSTM with categorical embeddings.  
**Result:** val F1 ≈ 0.71 — significantly worse than the best model; PCA discards temporal structure needed by the LSTM.

### Pipeline
1. Drop zero-variance `joint_30`
2. Min-max normalise the 30 joint-angle columns
3. PCA (retain 90 % variance) → 6 principal components
4. Embed categorical features (`n_legs`, `n_hands`, `n_eyes`) via `nn.Embedding`
5. Bidirectional LSTM classifier with weighted cross-entropy loss

> **Note:** This notebook serves as a baseline showing why raw PCA is suboptimal for temporal data.
  The PCA is fitted only on the training split and applied to val/test via `transform`.

In [ ]:
#Libraries import

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # Plotting 
import seaborn as sns # Statistical data visualization
import warnings # To suppress warnings
import torch 
from torch import nn #Neural Networks 
from torch.utils.data import TensorDataset, DataLoader
import random
import logging   
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix  # Usefull metrics
from sklearn.model_selection import train_test_split
import re           # Regex
from sklearn.decomposition import PCA
import math

In [ ]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
SEED = 42 # To guarantee reproducibility
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Set torch options

logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models


# Use a GPU if available, or use a CPU instead (N.B. GPUs are made available from kaggle)
if torch.cuda.is_available():     
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

# Check torch version 
print(f"PyTorch version: {torch.__version__}")


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

In [ ]:
#Set environment variables

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

In [ ]:
#Suppress warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

In [ ]:
#Data Loading

X = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train.csv')
y = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_train_labels.csv')
X_test = pd.read_csv('/kaggle/input/pirate-pain/pirate_pain_test.csv')

In [ ]:
print(X['joint_30'].unique())
print(X_test['joint_30'].unique())

In [ ]:
# Drop the last joint column, since it has always the same value
X.drop('joint_30', axis=1, inplace=True)
X_test.drop('joint_30', axis=1, inplace=True)

In [ ]:
# PCA to retain 90% of the variance of the joint columns
pca = PCA(0.9)

In [ ]:
# How many samples?
samples = X['sample_index'].unique()
n_samples = len(samples) 
print(n_samples) # 661

In [ ]:
random.shuffle(samples)

In [ ]:
print(samples[:10])

In [ ]:
N_VAL_SAMPLES = 60 # Number of samples used for validation (tunable parameter)

n_train_samples = len(samples) - N_VAL_SAMPLES

print(n_train_samples)

In [ ]:
# Split the shuffled samples id into training and validation
train_samples = samples[:n_train_samples]
val_samples = samples[n_train_samples:n_train_samples + N_VAL_SAMPLES]

In [ ]:
# Separate categorical data for future embedding

categorical_features = ['sample_index', 'time', 'n_legs', 'n_hands', 'n_eyes']
X_cat = X[categorical_features]
X_test_cat = X_test[categorical_features]

In [ ]:
X_non_cat = X.drop(['n_legs', 'n_hands', 'n_eyes'], axis=1)
X_test_non_cat = X_test.drop(['n_legs', 'n_hands', 'n_eyes'], axis=1)

In [ ]:
# Split the dataset into training and validation sets based on samples ids
X_train_cat = X_cat[X_cat['sample_index'].isin(train_samples)]
X_val_cat = X_cat[X_cat['sample_index'].isin(val_samples)]
X_train_non_cat = X_non_cat[X_non_cat['sample_index'].isin(train_samples)]
X_val_non_cat = X_non_cat[X_non_cat['sample_index'].isin(val_samples)]

In [ ]:
print(X_train_non_cat.shape)

In [ ]:
print(X_train_non_cat.head)

In [ ]:
y_train = y[y['sample_index'].isin(train_samples)]
y_val = y[y['sample_index'].isin(val_samples)]

In [ ]:
X_train_cat['n_legs'] = X_train_cat['n_legs'].replace('two', 'two_legs')
X_train_cat['n_hands'] = X_train_cat['n_hands'].replace('two', 'two_hands')
X_train_cat['n_eyes'] = X_train_cat['n_eyes'].replace('two', 'two_eyes')
print(X_train_cat.head())

X_val_cat['n_legs'] = X_val_cat['n_legs'].replace('two', 'two_legs')
X_val_cat['n_hands'] = X_val_cat['n_hands'].replace('two', 'two_hands')
X_val_cat['n_eyes'] = X_val_cat['n_eyes'].replace('two', 'two_eyes')
print(X_val_cat.head())

X_test_cat['n_legs'] = X_test_cat['n_legs'].replace('two', 'two_legs')
X_test_cat['n_hands'] = X_test_cat['n_hands'].replace('two', 'two_hands')
X_test_cat['n_eyes'] = X_test_cat['n_eyes'].replace('two', 'two_eyes')
print(X_test_cat.head())

In [ ]:
embedding_map = {
    'two_legs' : 0,
    'two_hands' : 1,
    'two_eyes' : 2,
    'one+peg_leg' : 3,
    'one+hook_hand' : 4,
    'one+eye_patch' : 5
}

In [ ]:
# Map categorical values to integer lables
X_train_cat['n_legs'] = X_train_cat['n_legs'].map(embedding_map)
X_train_cat['n_hands'] = X_train_cat['n_hands'].map(embedding_map)
X_train_cat['n_eyes'] = X_train_cat['n_eyes'].map(embedding_map)

print(X_train_cat.head())

In [ ]:
X_val_cat['n_legs'] = X_val_cat['n_legs'].map(embedding_map)
X_val_cat['n_hands'] = X_val_cat['n_hands'].map(embedding_map)
X_val_cat['n_eyes'] = X_val_cat['n_eyes'].map(embedding_map)

print(X_val_cat.head())

In [ ]:
X_test_cat['n_legs'] = X_test_cat['n_legs'].map(embedding_map)
X_test_cat['n_hands'] = X_test_cat['n_hands'].map(embedding_map)
X_test_cat['n_eyes'] = X_test_cat['n_eyes'].map(embedding_map)

print(X_test_cat.head())

In [ ]:
# Normalization
joint_cols = [col for col in X_train_non_cat.columns if re.match(r'joint_\d{2}', col)]

mins = X_train_non_cat[joint_cols].min()
maxs = X_train_non_cat[joint_cols].max()

for col in joint_cols:
    den = maxs[col] - mins[col]
    
    if den == 0:
        den = maxs[col]
        
    X_train_non_cat[col] = (X_train_non_cat[col] - mins[col])/den # Normalize train set
    X_val_non_cat[col] = (X_val_non_cat[col] - mins[col])/den # Normalize val set
    X_test_non_cat[col] = (X_test_non_cat[col] - mins[col])/den # Normalize test set

In [ ]:
print(X_train_non_cat.head)

In [ ]:
print(X_train_non_cat.shape)

In [ ]:
X_train_num = X_train_non_cat[joint_cols]
X_val_num = X_val_non_cat[joint_cols]
X_test_num = X_test_non_cat[joint_cols]

In [ ]:
print(X_train_num.shape)

In [ ]:
# Apply PCA
# Fit ONLY on training data; transform val/test to prevent data leakage
X_train_pca = pca.fit_transform(X_train_num)
X_val_pca   = pca.transform(X_val_num)
X_test_pca  = pca.transform(X_test_num)


In [ ]:
pca.n_components_

In [ ]:
# Rename PCA columns

train_pca = pd.DataFrame(X_train_pca, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6'])
X_train_non_cat = pd.concat([X_train_non_cat.drop(joint_cols, axis=1).reset_index(drop=True), train_pca.reset_index(drop=True)], axis=1 )

val_pca = pd.DataFrame(X_val_pca, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6'])
X_val_non_cat = pd.concat([X_val_non_cat.drop(joint_cols, axis=1).reset_index(drop=True), val_pca.reset_index(drop=True)], axis=1 )

test_pca = pd.DataFrame(X_test_pca, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6'])
X_test_non_cat = pd.concat([X_test_non_cat.drop(joint_cols, axis=1).reset_index(drop=True), test_pca.reset_index(drop=True)], axis=1 )

In [ ]:
print(train_pca.shape)

In [ ]:
print(X_train_non_cat.columns)

In [ ]:
print(X_train_non_cat.shape)

In [ ]:
print(X_train_non_cat.head)

In [ ]:
X_train_non_cat = X_train_non_cat[X_train_non_cat['sample_index'].notna()]
X_val_non_cat = X_val_non_cat[X_val_non_cat['sample_index'].notna()]
X_test_non_cat = X_test_non_cat[X_test_non_cat['sample_index'].notna()]

In [ ]:
print(X_train_non_cat.shape)

In [ ]:
training_labels = {
    'no_pain' : 0,
    'low_pain' : 0,
    'high_pain' : 0,
}


# Count occurrences of each activity for unique IDs in the training set
for id in y_train['sample_index'].unique():
    label = y_train[y_train['sample_index'] == id]['label'].values[0]
    training_labels[label] += 1

class_counts = torch.tensor([training_labels['no_pain'], training_labels['low_pain'], training_labels['high_pain']], dtype=torch.float)

In [ ]:
print(class_counts)

In [ ]:
print(list(X_train_non_cat.columns))

In [ ]:
data_cols = [col for col in X_train_non_cat.columns]
for cols in list(X_train_cat.columns):
    data_cols.append(cols)

In [ ]:
data_cols.remove('sample_index')
data_cols.remove('time')
print(data_cols)

In [ ]:
data_cols_non_cat = [col for col in data_cols if not col in ['n_legs', 'n_hands', 'n_eyes']]
print(data_cols_non_cat)

In [ ]:
# Mapping labels to integer data

label_map = {
    'no_pain' : 0,
    'low_pain' : 1,
    'high_pain' : 2
}

y_train['label'] = y_train['label'].map(label_map)
y_val['label'] = y_val['label'].map(label_map)

print(y_train[:10])
print(y_val[:10])

In [ ]:
y['label'] = y['label'].map(label_map)

In [ ]:
print(y)

In [ ]:
 # Window size and stride for sequence building

WINDOW_SIZE = 20 
STRIDE = 5 # the window need to be divisible by the stride

In [ ]:
# Function to build sequences

def build_sequences(df, window, stride, isTest, data_cols):
    # Be sure that the window is divisible by the stride
    assert window%stride == 0

    #Lists to store sequences and corresponding labels
    dataset = []
    labels = []

    # Iterate over the ids
    for id in df['sample_index'].unique():
        id_int = int(id)
        # Extract data from the current id
        temp = df[df['sample_index'] == id_int][data_cols].values

        if(not isTest):
            # Retrive the label for the id
            label = y[y['sample_index'] == id_int]['label'].values[0]

        # Calculate padding length to ensure full windows
        padding_len = window - len(temp) % window

        # Create zero padding and concatenate with the data
        padding = np.zeros((padding_len, len(data_cols)), dtype='float32')
        temp = np.concatenate((temp, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp):
            dataset.append(temp[idx:idx + window])
            if(not isTest):
                labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    if(not isTest):
        labels = np.array(labels)

    if(not isTest):
        return dataset, labels
    else:
        return dataset


In [ ]:
X_train_seq_cat, y_train_seq_cat = build_sequences(X_train_cat, WINDOW_SIZE, STRIDE, False, ['n_legs', 'n_hands', 'n_eyes'])
X_val_seq_cat, y_val_seq_cat = build_sequences(X_val_cat, WINDOW_SIZE, STRIDE, False, ['n_legs', 'n_hands', 'n_eyes'])
X_test_seq_cat = build_sequences(X_test_cat, WINDOW_SIZE, STRIDE, True, ['n_legs', 'n_hands', 'n_eyes'])

In [ ]:
# Define the number of classes based on the categorical labels
num_classes = len(np.unique(y_train_seq_cat))
print(num_classes)

In [ ]:
X_train_seq_non_cat, y_train_seq_non_cat = build_sequences(X_train_non_cat, WINDOW_SIZE, STRIDE, False, data_cols_non_cat)
X_val_seq_non_cat, y_val_seq_non_cat = build_sequences(X_val_non_cat, WINDOW_SIZE, STRIDE, False, data_cols_non_cat)
X_test_seq_non_cat = build_sequences(X_test_non_cat, WINDOW_SIZE, STRIDE, True, data_cols_non_cat)

In [ ]:
train_ds_cat = TensorDataset(torch.from_numpy(X_train_seq_cat), torch.from_numpy(X_train_seq_non_cat), torch.from_numpy(y_train_seq_cat))
val_ds_cat   = TensorDataset(torch.from_numpy(X_val_seq_cat), torch.from_numpy(X_val_seq_non_cat) ,torch.from_numpy(y_val_seq_cat))
test_ds_cat = TensorDataset(torch.from_numpy(X_test_seq_cat), torch.from_numpy(X_test_seq_non_cat))

In [ ]:
print(X_train_seq_non_cat.shape)

In [ ]:
print(train_ds_cat.tensors[0].shape)
print(train_ds_cat.tensors[1].shape)
print(train_ds_cat.tensors[2].shape)

In [ ]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [ ]:
BATCH_SIZE = 512

In [ ]:
train_cat_loader = make_loader(train_ds_cat, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_cat_loader   = make_loader(val_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_cat_loader   = make_loader(test_ds_cat, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
# Training configuration
LEARNING_RATE = 1e-3
EPOCHS = 512
PATIENCE = 50

# Architecture
HIDDEN_LAYERS = 2        # Hidden layers
HIDDEN_SIZE = 128        # Neurons per layer

# Regularisation
DROPOUT_RATE = 0.2         # Dropout probability
L1_LAMBDA = 0            # L1 penalty
L2_LAMBDA = 0            # L2 penalty

In [ ]:
weights = 1/class_counts  # To give more importance to misclassifications of rare classes
weights = weights/weights.mean() # weights normalizations
print(weights)
weights = weights.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
class RecurrentClassifierWithEmbedding(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_layers,
        num_classes,
        rnn_type,
        bidirectional=False,
        dropout_rate=0.2,
        vocab=6,
        embedding_dim=3,
        numerical_feat = len(data_cols_non_cat)
    ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        input_size = int(numerical_feat + embedding_dim*(vocab/2))

        self.embedding = nn.Embedding(vocab, embedding_dim)
        
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', 'GRU'")

        rnn_module = rnn_map[rnn_type]

        dropout_val = dropout_rate if num_layers > 1 else 0

        # Recurrent Layer
        self.rnn = rnn_module(
            input_size =  input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True, # Input shape: (batch, seq_len, features)
            bidirectional = bidirectional,
            dropout = dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    
    def forward(self, x_cat, x_non_cat):

        x_cat_emb = self.embedding(x_cat.long())

        x_cat_emb = x_cat_emb.view(x_cat_emb.size(0), x_cat_emb.size(1), -1) # flatten the last two dimenions
    

        x = torch.cat((x_non_cat, x_cat_emb), dim=-1) # Concatenate the categorical and numerical part

        
        # rnn_out shape: (batch_size, seq_len, hidden_size * num_features)
        rnn_out, hidden = self.rnn(x)    # rnn_out and hidden are the same thing, it's just torch preference to output them both

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':       # h_n = hidden state, c_n = cell state
            hidden = hidden[0]

        # hidden shape: (num_layers * num_features, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        return logits

In [ ]:
def train_one_epoch_cat(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
  
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs_cat, inputs_non_cat, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs_cat, inputs_non_cat, targets = inputs_cat.to(device).float(), inputs_non_cat.to(device).float(), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs_cat, inputs_non_cat)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs_cat.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [ ]:
def validate_one_epoch_cat(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs_cat, inputs_non_cat, targets in val_loader:
            # Move data to device
            inputs_cat, inputs_non_cat, targets = inputs_cat.to(device).float(), inputs_non_cat.to(device).float(), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs_cat, inputs_non_cat)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs_cat.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [ ]:
def fit_cat(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric='val_f1', mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name=''):

    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    if patience > 0:
        patience_counter = 0
        best_metric = float('-inf') if mode == 'max' else float('inf')
        best_epoch = 0

    print(f'Training {epochs} epochs...')

    for epoch in range(1, epochs + 1):

        train_loss, train_f1 = train_one_epoch_cat(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
        val_loss, val_f1 = validate_one_epoch_cat(
            model, val_loader, criterion, device
        )

        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        if verbose > 0 and (epoch % verbose == 0 or epoch == 1):
            print(f'Epoch {epoch:3d}/{epochs} | '
                  f'Train: Loss={train_loss:.4f}, F1={train_f1:.4f} | '
                  f'Val: Loss={val_loss:.4f}, F1={val_f1:.4f}')

        if patience > 0:
            current_metric = training_history[evaluation_metric][-1]
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                torch.save(model.state_dict(), 'models/' + experiment_name + '_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f'Early stopping triggered after {epoch} epochs.')
                    break

    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load('models/' + experiment_name + '_model.pt'))
        print(f'Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}')

    if patience == 0:
        torch.save(model.state_dict(), 'models/' + experiment_name + '_model.pt')

    return model, training_history


In [ ]:
# Create model and display architecture with parameter count
lstm_pca = RecurrentClassifierWithEmbedding(
    hidden_size=HIDDEN_SIZE,
    num_layers=HIDDEN_LAYERS,
    num_classes=num_classes,
    dropout_rate=DROPOUT_RATE,
    bidirectional=True,
    rnn_type='LSTM',
    embedding_dim=3,
    ).to(device)


# Define optimizer with L2 regularization
optimizer = torch.optim.AdamW(lstm_pca.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

# Enable mixed precision training for GPU acceleration
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [ ]:
%%time

model_path = '/kaggle/working/models/lstm_pca.pth'

if(not os.path.exists(model_path)):
    # Train model and track training history
    lstm_pca, training_history = fit_cat(
        model=lstm_pca,
        train_loader=train_cat_loader,
        val_loader=val_cat_loader,
        epochs=EPOCHS,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        verbose=1,
        experiment_name="lstm_emb",
        patience=PATIENCE
        )
    
    # Update best model if current performance is superior
    if training_history['val_f1'][-1] > best_performance:
        best_model = lstm_pca
        best_performance = training_history['val_f1'][-1]

    # Save model
    torch.save(lstm_pca.state_dict(), model_path)
else:
    # Load the model
    lstm_pca.load_state_dict(torch.load(model_path))

In [ ]:
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

In [ ]:
# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb_cat, xb_non_cat, yb in val_cat_loader:
        xb_cat = xb_cat.to(device).float()
        xb_non_cat = xb_non_cat.to(device).float()

        # Forward pass: get model predictions
        logits = lstm_pca(xb_cat, xb_non_cat)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# Calculate overall validation metrics
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

In [ ]:
# Set model to evaluation mode
lstm_pca.eval()

# Lists to store predictions
sample_indices = []
predictions = []

# Mapping from class index to label name
idx_to_label = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

# Generate predictions
with torch.no_grad():  # Disable gradient computation for inference

    sample_count=0

    
    for batch_idx, (batch_cat, batch_non_cat) in enumerate(test_cat_loader):

        # batch is a list
        if isinstance(batch_cat, list):
            xb_cat = batch_cat[0]  # Get the first element (the features)
        else:
            xb_cat = batch_cat

        if isinstance(batch_non_cat, list):
            xb_non_cat = batch_non_cat[0]  # Get the first element (the features)
        else:
            xb_non_cat = batch_non_cat
        
        xb_cat = xb_cat.to(device).float()
        xb_non_cat = xb_non_cat.to(device).float()

        
        #Get model predictions
        logits = lstm_pca(xb_cat, xb_non_cat)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Convert predictions to labels and store
        for i, pred_idx in enumerate(preds):

            
            if(((batch_idx*BATCH_SIZE)+i) % (X_test_seq_cat.shape[0] / 1324)  == 0):    
                sample_indices.append(f"{sample_count:03d}")
                predictions.append(idx_to_label[pred_idx])
                sample_count += 1


submission_df = pd.DataFrame({
    'sample_index': sample_indices,
    'label': predictions
})


In [ ]:
os.makedirs('/kaggle/working/submissions')

In [ ]:
submission_df.to_csv('/kaggle/working/submissions/submission_pca.csv', index=False)

In [ ]:
print(submission_df.head(10))

In [ ]:
print("\nPrediction distribution:")
print(submission_df['label'].value_counts())